# Lab 06: DCGAN on MNIST

**Module 06** | Read `notes/06-dcgan-gan-training.pdf` before running this notebook.

Apply DCGAN architectural rules (strided convolutions, batch norm, no pooling) and compare sample quality to the vanilla GAN lab.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Lab 06: DCGAN on MNIST

In Lab 05 you trained a simple GAN with fully connected layers. This lab upgrades the idea to **DCGAN** (Deep Convolutional GAN), which uses convolution layers so the network can see **spatial structure** in 28x28 digit images. The goal is the same: a **generator** fakes digits and a **discriminator** tries to tell real from fake.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **GAN (Generative Adversarial Network)** | Two neural networks compete: one creates fakes, one judges them. They improve each other through training. |
| **Generator (G)** | A network that turns random noise into an image. It learns to draw convincing digits. |
| **Discriminator (D)** | A network that scores an image as real (1) or fake (0). It learns to spot mistakes in generated images. |
| **Latent vector (noise)** | A list of random numbers (here 100 values) that the generator uses as a seed for each fake image. |
| **Transposed convolution** | A layer that upsamples a small feature map into a larger image. The generator uses these to grow noise into a 28x28 grid. |
| **Strided convolution** | A layer that shrinks spatial size while learning local patterns. The discriminator uses these to summarize an image. |
| **Batch normalization** | Keeps activations in a stable numeric range so training does not explode or stall. |
| **Adversarial training** | Alternating updates: D learns to classify better, then G learns to fool D. Neither side wins permanently. |
| **BCE loss** | Binary Cross-Entropy: a score that measures how wrong the discriminator's real/fake predictions are. |
| **Epoch** | One full pass through the entire training dataset. We run 6 epochs here. |
| **Tanh activation** | Squashes outputs to the range [-1, 1], which matches how we normalized MNIST pixels. |
| **Mode collapse** | A failure mode where the generator outputs the same few images repeatedly instead of diverse digits. |

Refer back to this table whenever a term appears in code or output.


### Step 1: Load and inspect MNIST

**What this does:** Downloads MNIST (if needed), scales pixels to [-1, 1], builds a DataLoader, and prints dataset statistics plus one real batch.

**Why we do this:** DCGAN training needs many real digit images as examples of what "real" looks like. Normalizing to [-1, 1] matches the generator's Tanh output range so real and fake images live on the same scale.


**What to look for in the output**

Train images should be about 60,000. Image shape should be `(1, 28, 28)` (one channel, 28 pixels tall and wide). Pixel min/max on a real batch should be near -1.0 and 1.0 after normalization. Batches per epoch should be roughly 468 (60,000 / 128).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

# --- Paths and hyperparameters ---
ROOT = Path("..").resolve()  # repo root (one folder above labs/)
DATA_DIR = ROOT / "datasets" / "mnist"
BATCH_SIZE = 128  # images processed together per training step
LATENT_DIM = 100  # length of the random noise vector fed to the generator
EPOCHS = 6
LR = 2e-4  # learning rate for Adam optimizer

# --- Preprocessing: tensor in [0,1], then scale to [-1, 1] for Tanh compatibility ---
transform = transforms.Compose([
    transforms.ToTensor(),  # PIL image -> float tensor in [0, 1]
    transforms.Normalize((0.5,), (0.5,)),  # (x - 0.5) / 0.5 maps [0,1] to [-1,1]
])

# --- Download/load MNIST training split ---
train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print("Dataset summary")
print(f"  Train images: {len(train_set):,}")
print(f"  Image shape:  {tuple(train_set[0][0].shape)}")
print(f"  Batches/epoch: {len(loader)}")
print(f"  Latent dim:   {LATENT_DIM}")

# --- Peek at one batch before moving to GPU ---
real_batch, real_labels = next(iter(loader))
print("\nOne real batch (before .to(device))")
print(f"  Batch shape:  {tuple(real_batch.shape)}")
print(f"  Pixel min/max: {real_batch.min():.3f} / {real_batch.max():.3f}")
print(f"  Pixel mean/std: {real_batch.mean():.3f} / {real_batch.std():.3f}")
print(f"  First 16 labels: {real_labels[:16].tolist()}")


### Step 2: Define generator and discriminator

**What this does:** Builds two `nn.Module` classes. The generator upsamples noise through transposed convolutions. The discriminator downsamples images through strided convolutions and outputs one logit per image.

**Why we do this:** Convolution layers respect the 2D grid structure of digits (strokes, curves, loops). DCGAN design rules (batch norm, LeakyReLU, no pooling, no big fully connected layers) make GAN training more stable than the shallow MLP from Lab 05.


**What to look for in the output**

No output is printed here. After running, the classes `DCGANGenerator` and `DCGANDiscriminator` should exist in memory. The generator crops a 32x32 internal map down to 28x28 to match MNIST.


In [ ]:
class DCGANGenerator(nn.Module):
    """Maps a noise vector to a 28x28 grayscale digit image."""

    def __init__(self, latent_dim: int = 100, ngf: int = 64):
        super().__init__()
        # Transposed convolutions grow spatial size: 1x1 -> 4x4 -> 8x8 -> ... -> 32x32
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, 1, 4, 2, 1, bias=False),
            nn.Tanh(),  # output pixels in [-1, 1]
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # z shape: (batch, latent_dim) -> reshape to (batch, latent_dim, 1, 1)
        x = self.net(z.view(z.size(0), -1, 1, 1))
        # Center-crop 32x32 down to 28x28 MNIST size
        return x[:, :, 2:30, 2:30]


class DCGANDiscriminator(nn.Module):
    """Scores each image with one logit: high = looks real, low = looks fake."""

    def __init__(self, ndf: int = 64):
        super().__init__()
        # Strided convolutions shrink 28x28 -> smaller feature maps
        self.net = nn.Sequential(
            nn.Conv2d(1, ndf, 4, 2, 1, bias=False),  # no BN on first layer (DCGAN rule)
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 1, 1, 0, bias=False),  # 1x1 conv on 1x1 map (28x28 MNIST)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(-1)  # flatten to (batch,) logits


### Step 3: Instantiate models and optimizers

**What this does:** Creates G and D on `device`, sets up BCE loss and Adam optimizers, counts parameters, and runs a quick forward pass on random noise.

**Why we do this:** A sanity check before training confirms tensor shapes and that both networks can run on your CPU or GPU. Parameter counts help you understand model size.


**What to look for in the output**

Generator should have on the order of 3 million parameters; discriminator roughly 2.7 million. Fake image shape should be `(4, 1, 28, 28)`. Discriminator logits should be a list of 4 floating-point numbers (untrained, so they are random-looking).


In [ ]:
# Move both networks to the active device (CPU or CUDA from the device cell)
G = DCGANGenerator(LATENT_DIM).to(device)
D = DCGANDiscriminator().to(device)

# BCEWithLogitsLoss combines sigmoid + BCE in one numerically stable op
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

# Count learnable weights
g_params = sum(p.numel() for p in G.parameters())
d_params = sum(p.numel() for p in D.parameters())
print("Architecture parameter counts")
print(f"  Generator:      {g_params:,}")
print(f"  Discriminator:  {d_params:,}")
print(f"  Total:          {g_params + d_params:,}")

# Forward pass without training (no_grad) to verify shapes
with torch.no_grad():
    z_test = torch.randn(4, LATENT_DIM, device=device)  # 4 random noise vectors
    fake_test = G(z_test)
    d_out = D(fake_test)
print("\nSanity check (batch of 4)")
print(f"  Fake image shape: {tuple(fake_test.shape)}")
print(f"  Discriminator logits: {d_out.cpu().tolist()}")


### Step 4: Train the DCGAN

**What this does:** Runs the adversarial loop for 6 epochs. Each batch: update D on real and fake images, then update G to fool D. Tracks losses and discriminator accuracy.

**Why we do this:** This is the core learning process. D must not become perfect (100% on fakes), or G stops learning. Healthy training often shows D accuracy around 50-70% on both real and fake.


**What to look for in the output**

Each epoch prints G loss, D loss, D acc real, and D acc fake. Losses should move but not explode. If D acc fake hits 100% every epoch, the generator is struggling. Training takes several minutes on CPU.


In [ ]:
def batch_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    # Convert logits to probabilities, threshold at 0.5, compare to 0/1 labels
    preds = (torch.sigmoid(logits) > 0.5).float()
    return (preds == labels).float().mean().item()


g_losses, d_losses = [], []
d_real_accs, d_fake_accs = [], []
# Fixed noise lets us compare sample quality across epochs (used in Step 6)
fixed_noise = torch.randn(64, LATENT_DIM, device=device)

for epoch in range(1, EPOCHS + 1):
    g_epoch, d_epoch = 0.0, 0.0
    real_acc_epoch, fake_acc_epoch = 0.0, 0.0
    for real, _ in loader:
        real = real.to(device)
        batch = real.size(0)
        real_labels = torch.ones(batch, device=device)   # target 1 = real
        fake_labels = torch.zeros(batch, device=device)  # target 0 = fake

        # --- Train discriminator ---
        D.zero_grad()
        d_real = D(real)
        loss_d_real = criterion(d_real, real_labels)
        z = torch.randn(batch, LATENT_DIM, device=device)
        fake = G(z).detach()  # detach: do not backprop into G when training D
        d_fake = D(fake)
        loss_d_fake = criterion(d_fake, fake_labels)
        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        opt_d.step()

        # --- Train generator (goal: make D output 1 for fakes) ---
        G.zero_grad()
        z = torch.randn(batch, LATENT_DIM, device=device)
        fake = G(z)
        d_out = D(fake)
        loss_g = criterion(d_out, real_labels)  # G wants D to think fakes are real
        loss_g.backward()
        opt_g.step()

        g_epoch += loss_g.item()
        d_epoch += loss_d.item()
        real_acc_epoch += batch_accuracy(d_real, real_labels)
        fake_acc_epoch += batch_accuracy(d_fake, fake_labels)

    g_losses.append(g_epoch / len(loader))
    d_losses.append(d_epoch / len(loader))
    d_real_accs.append(real_acc_epoch / len(loader))
    d_fake_accs.append(fake_acc_epoch / len(loader))
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"G loss {g_losses[-1]:.4f} | D loss {d_losses[-1]:.4f} | "
        f"D acc real {d_real_accs[-1]:.1%} | D acc fake {d_fake_accs[-1]:.1%}"
    )


### Step 5: Plot training curves

**What this does:** Plots generator vs discriminator loss and discriminator accuracy on real vs fake batches over epochs.

**Why we do this:** Curves summarize the adversarial balance. Rising G loss with falling D loss can signal mode collapse or a discriminator that is too strong.


**What to look for in the output**

Two subplots should appear. Loss curves may wiggle; that is normal for GANs. Accuracy curves near 50% on both classes suggest a balanced game.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(range(1, EPOCHS + 1), g_losses, label="Generator", marker="o")
axes[0].plot(range(1, EPOCHS + 1), d_losses, label="Discriminator", marker="s")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE loss")
axes[0].set_title("DCGAN training losses")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, EPOCHS + 1), d_real_accs, label="Real accuracy", marker="o")
axes[1].plot(range(1, EPOCHS + 1), d_fake_accs, label="Fake accuracy", marker="s")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Discriminator accuracy")
axes[1].set_title("Discriminator classification accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Step 6: Generate samples from fixed noise

**What this does:** Uses the same 64 noise vectors every time (`fixed_noise`) to generate an 8x8 grid of fake digits and prints pixel statistics.

**Why we do this:** Fixed noise makes it easy to see quality improve across training runs. You can visually judge whether outputs look like digits or gray mush.


**What to look for in the output**

Pixel mean near 0 and std below 1.0 are typical for Tanh outputs. The grid should show digit-like blobs after 6 epochs (not perfect, but structured).


In [ ]:
G.eval()  # inference mode (no dropout/batchnorm randomness)
with torch.no_grad():
    fake_batch = G(fixed_noise).cpu()

print("Fixed-noise sample batch")
print(f"  Shape: {tuple(fake_batch.shape)}")
print(f"  Pixel min/max: {fake_batch.min():.3f} / {fake_batch.max():.3f}")
print(f"  Pixel mean/std: {fake_batch.mean():.3f} / {fake_batch.std():.3f}")
print("  Interpretation: mean near 0 and std below 1.0 are typical for Tanh outputs.")

# make_grid stitches 64 images into one mosaic; normalize maps [-1,1] to [0,1] for display
grid = make_grid(fake_batch, nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("DCGAN samples (64 fixed noise vectors)")
plt.axis("off")
plt.tight_layout()
plt.show()
